In [ ]:
import os
import glob
import json
import sqlite3
import uuid
from datetime import datetime
import pandas as pd

# Percorsi di lettura e scrittura coerenti con il setup del database
PARQUET_DIR = os.path.join("..", "data", "raw")
DB_PATH = os.path.join("..", "data", "platformhero_mirror.db")
print(f" Leggendo i Parquet da: {PARQUET_DIR}")
print(f" Scrivendo nel Database: {DB_PATH}")

In [ ]:
def parsing_conversazione_to_markdown(conversation):
    if conversation is None:
        return ""
    
    # Se è un array NumPy, lo trasformiamo in una lista Python nativa
    if hasattr(conversation, "tolist"):
        conversation = conversation.tolist()
        
    # Se per qualche motivo è una stringa JSON, ne facciamo il parsing
    if isinstance(conversation, str):
        try:
            conversation = json.loads(conversation)
        except Exception:
            return conversation
            
    markdown_text = ""
    
    # Generazione pulita del testo in formato Markdown
    if isinstance(conversation, list):
        for turn in conversation:
            if isinstance(turn, dict):
                role = str(turn.get("role", "unknown")).capitalize()
                content = turn.get("content", "")
                markdown_text += f"### {role}\n{content}\n\n"
                
    return markdown_text.strip()

In [ ]:
# Apertura connessione al database specchio
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Recupero controllato di tutti i file parquet
parquet_files = sorted(glob.glob(os.path.join(PARQUET_DIR, "*.parquet")))
print(f"🔍 Trovati {len(parquet_files)} file Parquet da elaborare.\n")

for idx, file_path in enumerate(parquet_files):
    file_name = os.path.basename(file_path)
    print(f" [{idx+1}/{len(parquet_files)}] Elaborazione in corso di: {file_name}...")
    
    df_chunk = pd.read_parquet(file_path)
    rows_to_insert = [] # Svuotiamo la lista a ogni file per salvare la memoria
    
    for _, row in df_chunk.iterrows():
        internal_id = str(uuid.uuid4())
        external_id = row["conversation_id"]
        
        ticket = f"#LMSYS-{external_id[:5].upper()}"
        client = "LMSYS Open Dataset Client"
        area = row.get("model", "General LLM")
        category = f"Dialogue ({str(row.get('language', 'unknown')).capitalize()})"
        
        # Ora questa funzione estrarrà correttamente il testo dall'array NumPy!
        raw_markdown = parsing_conversazione_to_markdown(row["conversation"])
        
        moderation_labels = row.get("openai_moderation")
        if hasattr(moderation_labels, "tolist"):
            moderation_labels = moderation_labels.tolist()
            
        structured_data = {
            "turns_count": int(row.get("turn", 1)),
            "is_redacted": bool(row.get("redacted", False)),
            "moderation_labels": moderation_labels
        }
        structured_json = json.dumps(structured_data, ensure_ascii=False)
        
        usage_data = {
            "total_cost": round(0.0015 * int(row.get("turn", 1)), 5),
            "input_tokens": 120 * int(row.get("turn", 1)),
            "output_tokens": 150 * int(row.get("turn", 1))
        }
        usage_json = json.dumps(usage_data, ensure_ascii=False)
        
        created_at = datetime.utcnow().isoformat() + "Z"
        
        rows_to_insert.append((
            internal_id, external_id, ticket, client, area, 
            category, raw_markdown, structured_json, usage_json, created_at
        ))

    # SALVATAGGIO PARZIALE IMMEDIATO
    print(f"    Scrittura di {len(rows_to_insert)} record nel database...")
    cursor.executemany("""
        INSERT OR REPLACE INTO logs_normalized 
        (id, external_id, ticket, client, area, category, raw_markdown, structured_json, usage_json, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, rows_to_insert)
    
    conn.commit() # Rilascia il file su disco e svuota la RAM
    print(f" File {file_name} elaborato e salvato!\n")

print(" Sincronizzazione completata con successo per tutti i log!")

In [ ]:
# Lettura rapida di controllo tramite Pandas
df_check = pd.read_sql_query("SELECT ticket, area, category FROM logs_normalized LIMIT 5;", conn)

# Chiusura della connessione in sicurezza per liberare il file .db
conn.close()
print(" Connessione al database SQLite chiusa correttamente.")

display(df_check)